<a href="https://colab.research.google.com/github/DaniloDuque/logistic-regression/blob/main/src/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Regresión Logística

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')
    os.system('git pull')

sys.path.insert(0, os.path.abspath('src'))

---
# Sección 1 — Algoritmo de Regresión Logística

## 1.a — Usar el código del perceptrón como base
DOCUMENTACION EN LATEX: Explique en el reporte cada una de las modificaciones necesarias al código del perceptrón, utilizando como referencia las diferencias entre ambos algoritmos.

### Pruebas unitarias
2 pruebas unitarias (con su objetivo, entradas, salidas esperadas y resultados) para las funciones modificadas en el algoritmo del perceptron base.

---
## 1.b — Experimentos: Datos separables vs no separables

In [ ]:
import torch
from pathlib import Path

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_single_result, print_runs_table, compute_accuracy
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

FIGURES_DIR = Path('..') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

### Experimento 1 — Datos linealmente separables

Dos clases generadas con `cluster_std=1.0` (grupos compactos y bien separados).
Se espera que el modelo converja rápidamente y obtenga un MAE bajo.

In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
train_errors_s, test_errors_s = train_with_history(
    model_s, X_train_s, y_train_s, X_test_s, y_test_s, steps=STEPS, alpha=ALPHA
)

print(f"Iteraciones: {STEPS}")
print_single_result(
    'Linealmente separable',
    compute_mae(y_train_s, model_s.forward(X_train_s)),
    compute_mae(y_test_s,  model_s.forward(X_test_s)),
    compute_accuracy(model_s, X_train_s, y_train_s),
    compute_accuracy(model_s, X_test_s,  y_test_s),
)

plot_results(model_s, X_train_s, y_train_s, train_errors_s, test_errors_s,
             'Experimento 1 — Datos linealmente separables',
             output_path=FIGURES_DIR / 'experimento1.pdf')

### Experimento 2 — Datos no linealmente separables

Dos clases generadas con `cluster_std=4.0` (grupos solapados).
Se espera que el modelo tenga dificultades y obtenga un MAE más alto.

In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
train_errors_ns, test_errors_ns = train_with_history(
    model_ns, X_train_ns, y_train_ns, X_test_ns, y_test_ns, steps=STEPS, alpha=ALPHA
)

print(f"Iteraciones: {STEPS}")
print_single_result(
    'No linealmente separable',
    compute_mae(y_train_ns, model_ns.forward(X_train_ns)),
    compute_mae(y_test_ns,  model_ns.forward(X_test_ns)),
    compute_accuracy(model_ns, X_train_ns, y_train_ns),
    compute_accuracy(model_ns, X_test_ns,  y_test_ns),
)

plot_results(model_ns, X_train_ns, y_train_ns, train_errors_ns, test_errors_ns,
             'Experimento 2 — Datos no linealmente separables',
             output_path=FIGURES_DIR / 'experimento2.pdf')

### 1.2 — 10 ejecuciones: MAE medio, precisión, mínimo y máximo

In [ ]:
results_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
results_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)
print_runs_table(results_s, results_ns)

---
# Sección 2 — Regresión Logística y LLMs para clasificación de textos

## Pre-requisitos

In [ ]:
import torch
import nltk

from dataset import load_feina

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# df: DataFrame completo | all_texts: Segment + Proposal | all_labels: 1=complejo, 0=simple
df, all_texts, all_labels = load_feina()

STEPS_LR = 3000

# Sección 2.1 — Regresión logística con TF-IDF

In [ ]:
from tfidf import build_tfidf, transform_tfidf, display_tfidfs

# a) Examen cualitativo del preprocesamiento
from tfidf import preprocess_text
print('── a) Examen cualitativo del preprocesamiento ──')
for text in all_texts[:3]:
    print(f'  ORIGINAL:  {text}')
    print(f'  PROCESADO: {preprocess_text(text)}')
    print()

In [ ]:
# b) Construcción y examen cualitativo de descriptores TF-IDF
tfidf_vectorizer, X_train_vectorized = build_tfidf(all_texts)

print('── b) Examen cualitativo de descriptores TF-IDF ──')
display_tfidfs(transform_tfidf(tfidf_vectorizer, all_texts[6000:6003]))

In [ ]:
import torch
import pandas as pd

# c) Transformación a tensor
X_tfidf_full = transform_tfidf(tfidf_vectorizer, all_texts).toarray()
print(f'── c) Forma de la matriz TF-IDF: {X_tfidf_full.shape} ──')

X = torch.tensor(X_tfidf_full, dtype=torch.float64)
t = torch.tensor(all_labels,   dtype=torch.float64).unsqueeze(1)

In [ ]:
from sklearn.model_selection import train_test_split
from logistic_regression import LogisticRegression

X_train, X_test, t_train, t_test = train_test_split(X, t, test_size=0.2, random_state=42)

model_tfidf = LogisticRegression(torch.zeros((X.shape[1], 1), dtype=torch.float64))
model_tfidf.train(X_train, t_train, steps=STEPS_LR)

In [ ]:
# d) Resultados
print(f'  Accuracy entrenamiento: {model_tfidf.accuracy(X_train, t_train):.4f}')
print(f'  Accuracy prueba:        {model_tfidf.accuracy(X_test,  t_test):.4f}')

# Sección 2.2 — Regresión logística con vectores empotrados de BERT

In [ ]:
import torch
from embeddings import load_bert_models, get_embeddings_batch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# a) Calcular embeddings con BERT y RoBERTa
models = load_bert_models()
tokenizer_bert,    model_bert    = models['bert']
tokenizer_roberta, model_roberta = models['roberta']

emb_bert    = get_embeddings_batch(all_texts, model_bert,    tokenizer_bert,    device=device)
emb_roberta = get_embeddings_batch(all_texts, model_roberta, tokenizer_roberta, device=device)

In [ ]:
from logistic_regression import LogisticRegression

STEPS_LR = 10000
t_all = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)

# b) Entrenar con BERT
X_bert = torch.tensor(emb_bert, dtype=torch.float64)
model_bert_lr = LogisticRegression(torch.zeros((X_bert.shape[1], 1), dtype=torch.float64))
model_bert_lr.train(X_bert, t_all, STEPS_LR)

# Entrenar con RoBERTa
X_roberta = torch.tensor(emb_roberta, dtype=torch.float64)
model_roberta_lr = LogisticRegression(torch.zeros((X_roberta.shape[1], 1), dtype=torch.float64))
model_roberta_lr.train(X_roberta, t_all, STEPS_LR)

In [ ]:
# c) Evaluación sobre el conjunto de entrenamiento
print(f'Accuracy BERT:    {model_bert_lr.accuracy(X_bert,    t_all):.4f}')
print(f'Accuracy RoBERTa: {model_roberta_lr.accuracy(X_roberta, t_all):.4f}')

---
# Sección 2.3 — Modelos LLM para clasificación de complejidad
Modelos: `Recognai/zeroshot_selectra_medium` (español) y `facebook/bart-large-mnli`

In [ ]:
from llm_classifier import (
    load_classifiers, classify_all,
    show_examples, compute_llm_metrics, print_metrics_table
)

classifiers = load_classifiers()

ejemplos = [
    'El banco ofrece una cuenta de ahorros con interés anual del 3%.',
    'La tasa de rentabilidad ajustada por riesgo refleja la volatilidad implícita del subyacente en el mercado de derivados.',
    'Puedes guardar tu dinero en el banco y ganar un pequeño porcentaje extra.',
]
show_examples(classifiers, ejemplos)

predictions = classify_all(classifiers, all_texts)
metrics     = compute_llm_metrics(predictions, all_labels)
print_metrics_table(metrics)

# Sección 2.4 — Modelos LLM con few-shot

In [ ]:
from few_shot_classifier import (
    load_few_shot_classifiers, classify_all_configs,
    show_few_shot_examples, compute_few_shot_metrics, print_few_shot_metrics
)

fs_classifiers = load_few_shot_classifiers()

show_few_shot_examples(fs_classifiers, ejemplos)

fs_results = classify_all_configs(fs_classifiers, all_texts)
fs_metrics = compute_few_shot_metrics(fs_results, all_labels)
print_few_shot_metrics(fs_metrics)

---
# Sección 2.5 — Prueba completa: 30 corridas con partición 80/20

Tratamientos evaluados:
- TF-IDF + Regresión Logística
- BERT + Regresión Logística
- RoBERTa + Regresión Logística
- LLM Selectra (zero-shot)
- LLM BART (zero-shot)
- Few-Shot 1-shot / 3-shot / 5-shot (Selectra)

Pre-requisito: `X_tfidf_full`, `emb_bert`, `emb_roberta` calculados en secciones anteriores.

In [ ]:
from experiment import run_30_corridas, print_30_results, plot_30_results, print_summary

results_30 = run_30_corridas(
    all_texts, all_labels,
    X_tfidf=X_tfidf_full,
    emb_bert=emb_bert,
    emb_roberta=emb_roberta,
)

print_30_results(results_30)
plot_30_results(results_30)
print_summary(results_30)

# Sección 2.6 — Test de Friedman + Post-hoc Nemenyi

In [ ]:
!pip install scikit-posthocs -q

from experiment import friedman_nemenyi
friedman_nemenyi(results_30)